# Análise do Desempenho Olímpico e Características Socioeconômicas

**Grupo:** Erick Souza Alberti, Felipe Augusto da Silva Cruz e Guilherme Henrique Pereira Cunha

## Introdução



**Link do Repositório no GitHub:** https://github.com/felipeascruz/AEDV

## Descrição dos Dados

* **Olympic Athletes:** Repositório com informações de participações olímpicas (país, edição, esporte, atleta, medalha).
* **World Bank (API):** Indicadores anuais por país (população total, PIB e PIB per capita).

*(Explique também a metodologia de combinação das bases, focando no cruzamento pelas chaves de país e ano e possíveis tratamentos de identificadores.)*

In [8]:
# Transformar dados requeridos do World Bank em csv
import wbgapi as wb # API do World Bank

# Códigos oficiais das variáveis de interesse:
# SP.POP.TOTL: População total
# NY.GDP.MKTP.CD: PIB (US$ correntes)
# NY.GDP.PCAP.CD: PIB per capita (US$ correntes)
indicadores_wb = {
    'SP.POP.TOTL': 'population',
    'NY.GDP.MKTP.CD': 'gdp',
    'NY.GDP.PCAP.CD': 'gdp_per_capita'
}

# O dataset olímpico abrange de 1896 até 2026. A extração cobrirá esse período.
df_world_bank = wb.data.DataFrame(
    indicadores_wb.keys(),
    time=range(1896, 2027),
    labels=True,
    numericTimeKeys=True
).reset_index()

# Salva o CSV sem o índice numérico para evitar a coluna extra 'Unnamed: 0' e prevenir erros na leitura
df_world_bank.to_csv('world_bank.csv', index=False)

In [22]:
# Célula para importação de bibliotecas e leitura inicial dos dados
import pandas as pd
import altair as alt
from IPython.display import display

# Habilita o renderizador para Jupyter
alt.renderers.enable('default')

# Desabilita o limite de 5.000 colunas
alt.data_transformers.disable_max_rows()

df_medalhas = pd.read_csv('medal_table.csv')
df_atletas = pd.read_csv('olympic_athletes.csv')
df_world_bank = pd.read_csv('world_bank.csv')

# Transforma as colunas de anos em uma única coluna 'year'
df_world_bank = df_world_bank.melt(
    id_vars=['economy', 'series', 'Country', 'Series'], # Colunas mantidas fixas
    var_name='year', # Nova coluna para os anos
    value_name='valor' # Nova coluna para os números
)

# Pivota a tabela para que População, PIB e PIB per capita ganhem suas próprias colunas
df_world_bank = df_world_bank.pivot_table(
    index=['economy', 'Country', 'year'],
    columns='series',
    values='valor'
).reset_index()

df_world_bank['year'] = df_world_bank['year'].astype(int)

# Renomeia as colunas do World Bank para facilitar a junção com os dados olímpicos
df_world_bank = df_world_bank.rename(columns=indicadores_wb)
df_world_bank = df_world_bank.rename(columns={'economy': 'noc', 'Country': 'country'})

# Verifica quais anos do World Bank estão sem dados suficientes e exclui da tabela
# São considerados dados suficientes 2 das 3 colunas GDP (PIB), GDP Per Capita e População
df_world_bank = df_world_bank.dropna(subset=['population', 'gdp', 'gdp_per_capita'], thresh=2)

# Os códigos de países do comitê olímpico estão no padrão NOC, enquanto os do World Bank estão no padrão ISO. Transformaremos o data frame do World Bank para NOC.
iso_para_noc = {
    'DZA': 'ALG', 'AGO': 'ANG', 'ATG': 'ANT', 'ABW': 'ARU', 'ASM': 'ASA', 'BHS': 'BAH', 'BGD': 'BAN', 'BRB': 'BAR',
    'BMU': 'BER', 'BTN': 'BHU', 'BLZ': 'BIZ', 'BWA': 'BOT', 'BHR': 'BRN', 'BRN': 'BRU', 'BGR': 'BUL', 'BFA': 'BUR',
    'KHM': 'CAM', 'CYM': 'CAY', 'COG': 'CGO', 'TCD': 'CHA', 'CHL': 'CHI', 'CRI': 'CRC', 'HRV': 'CRO', 'DNK': 'DEN',
    'SLV': 'ESA', 'FJI': 'FIJ', 'GMB': 'GAM', 'GNB': 'GBS', 'GNQ': 'GEQ', 'DEU': 'GER', 'GRC': 'GRE', 'GRD': 'GRN',
    'GTM': 'GUA', 'GIN': 'GUI', 'HTI': 'HAI', 'HND': 'HON', 'IDN': 'INA', 'IRN': 'IRI', 'VIR': 'ISV', 'VGB': 'IVB',
    'XKX': 'KOS', 'SAU': 'KSA', 'KWT': 'KUW', 'LVA': 'LAT', 'LBY': 'LBA', 'LSO': 'LES', 'LBN': 'LIB', 'MDG': 'MAD',
    'MYS': 'MAS', 'MWI': 'MAW', 'MNG': 'MGL', 'MCO': 'MON', 'MUS': 'MRI', 'MRT': 'MTN', 'MMR': 'MYA', 'NIC': 'NCA',
    'NLD': 'NED', 'NPL': 'NEP', 'NGA': 'NGR', 'NER': 'NIG', 'OMN': 'OMA', 'PRY': 'PAR', 'PHL': 'PHI', 'PSE': 'PLE',
    'PRT': 'POR', 'PRI': 'PUR', 'ZAF': 'RSA', 'WSM': 'SAM', 'SYC': 'SEY', 'SGP': 'SIN', 'KNA': 'SKN', 'SVN': 'SLO',
    'SLB': 'SOL', 'LKA': 'SRI', 'SDN': 'SUD', 'CHE': 'SUI', 'TZA': 'TAN', 'TON': 'TGA', 'TGO': 'TOG', 'ARE': 'UAE',
    'URY': 'URU', 'VUT': 'VAN', 'VNM': 'VIE', 'VCT': 'VIN', 'ZMB': 'ZAM', 'ZWE': 'ZIM'
}
df_world_bank['noc'] = df_world_bank['noc'].replace(iso_para_noc)

correcoes_olimpicas = {
    'ROC': 'RUS', # Comitê Olímpico Russo -> Rússia
    'OAR': 'RUS', # Atletas Olímpicos da Rússia -> Rússia
    'EUN': 'URS'  # Equipe Unificada (ex-soviéticos em 1992) -> União Soviética
}
df_medalhas['noc'] = df_medalhas['noc'].replace(correcoes_olimpicas)

# 'display' é usado para que as duas tabelas sejam impressas
display(df_medalhas.head())
display(df_world_bank.head())

,edition_id,games,year,season,noc,country,gold,silver,bronze,total,notes
0,1,1896 Summer,1896,Summer,USA,United States,11,7,2,20,NaN
1,1,1896 Summer,1896,Summer,GRE,Greece,10,18,19,47,NaN
2,1,1896 Summer,1896,Summer,GER,Germany,6,5,2,13,NaN
3,1,1896 Summer,1896,Summer,FRA,France,5,4,2,11,NaN
4,1,1896 Summer,1896,Summer,GBR,Great Britain,2,3,2,7,NaN


series,noc,country,year,gdp,gdp_per_capita,population
26,ARU,Aruba,1986,4.055866e+08,6767.559229,59931.0
27,ARU,Aruba,1987,4.877095e+08,8244.045660,59159.0
28,ARU,Aruba,1988,5.966480e+08,10056.261393,59331.0
29,ARU,Aruba,1989,6.955307e+08,11507.217151,60443.0
30,ARU,Aruba,1990,7.648045e+08,12187.536361,62753.0


### 1. Como o desempenho olímpico dos países, medido pelo número de medalhas conquistadas, varia entre as diferentes edições dos Jogos analisadas?

**Transformação dos Dados:**

*(Documente aqui as agregações, derivações e limpezas feitas especificamente para responder a esta pergunta. Por exemplo, agrupar medalhas por equipe, ou derivar medalhas/milhão de habitantes)*

**Análise e Resposta:**

*(Interprete o gráfico gerado acima e responda diretamente à pergunta, lembrando de discutir em termos de associação e não causalidade.)*

### 2. Como as medalhas estão distribuídas entre os países participantes? Elas se concentram em um grupo relativamente pequeno de países?

**Transformação dos Dados:**

*(Documente aqui as agregações, derivações e limpezas feitas especificamente para responder a esta pergunta. Por exemplo, agrupar medalhas por equipe, ou derivar medalhas/milhão de habitantes)*

In [ ]:
# Código para transformação de dados e geração do gráfico para a 2ª Pergunta
# ...
# plt.show()

**Análise e Resposta:**

*(Interprete o gráfico gerado acima e responda diretamente à pergunta, lembrando de discutir em termos de associação e não causalidade.)*

### 3. Países com populações maiores tendem a conquistar mais medalhas?

**Transformação dos Dados:**

*(Documente aqui as agregações, derivações e limpezas feitas especificamente para responder a esta pergunta. Por exemplo, agrupar medalhas por equipe, ou derivar medalhas/milhão de habitantes)*

In [ ]:
# Código para transformação de dados e geração do gráfico para a 3ª Pergunta
# ...
# plt.show()

**Análise e Resposta:**

*(Interprete o gráfico gerado acima e responda diretamente à pergunta, lembrando de discutir em termos de associação e não causalidade.)*

### 4. Existe associação entre o PIB de um país e seu número de medalhas conquistadas?

**Transformação dos Dados:**

*(Documente aqui as agregações, derivações e limpezas feitas especificamente para responder a esta pergunta. Por exemplo, agrupar medalhas por equipe, ou derivar medalhas/milhão de habitantes)*

In [ ]:
# Código para transformação de dados e geração do gráfico para a 4ª Pergunta
# ...
# plt.show()

**Análise e Resposta:**

*(Interprete o gráfico gerado acima e responda diretamente à pergunta, lembrando de discutir em termos de associação e não causalidade.)*

### 5. A relação observada muda quando utilizamos PIB per capita em vez de PIB total?

**Transformação dos Dados:**

*(Documente aqui as agregações, derivações e limpezas feitas especificamente para responder a esta pergunta. Por exemplo, agrupar medalhas por equipe, ou derivar medalhas/milhão de habitantes)*

In [ ]:
# Código para transformação de dados e geração do gráfico para a 5ª Pergunta
# ...
# plt.show()

**Análise e Resposta:**

*(Interprete o gráfico gerado acima e responda diretamente à pergunta, lembrando de discutir em termos de associação e não causalidade.)*

### 6. Como a comparação entre os países muda quando utilizamos medidas relativas, como medalhas por milhão de habitantes, em vez do número absoluto de medalhas?

**Transformação dos Dados:**

*(Documente aqui as agregações, derivações e limpezas feitas especificamente para responder a esta pergunta. Por exemplo, agrupar medalhas por equipe, ou derivar medalhas/milhão de habitantes)*

In [ ]:
# Código para transformação de dados e geração do gráfico para a 6ª Pergunta
# ...
# plt.show()

**Análise e Resposta:**

*(Interprete o gráfico gerado acima e responda diretamente à pergunta, lembrando de discutir em termos de associação e não causalidade.)*

### 7. Quais países apresentam desempenho olímpico muito acima ou abaixo do padrão observado para países com características socioeconômicas semelhantes?

**Transformação dos Dados:**

*(Documente aqui as agregações, derivações e limpezas feitas especificamente para responder a esta pergunta. Por exemplo, agrupar medalhas por equipe, ou derivar medalhas/milhão de habitantes)*

In [ ]:
# Código para transformação de dados e geração do gráfico para a 7ª Pergunta
# ...
# plt.show()

**Análise e Resposta:**

*(Interprete o gráfico gerado acima e responda diretamente à pergunta, lembrando de discutir em termos de associação e não causalidade.)*

### 8. Essas relações entre características socioeconômicas e desempenho olímpico permanecem semelhantes ao longo das diferentes edições dos Jogos?

**Transformação dos Dados:**

*(Documente aqui as agregações, derivações e limpezas feitas especificamente para responder a esta pergunta. Por exemplo, agrupar medalhas por equipe, ou derivar medalhas/milhão de habitantes)*

In [ ]:
# Código para transformação de dados e geração do gráfico para a 8ª Pergunta
# ...
# plt.show()

**Análise e Resposta:**

*(Interprete o gráfico gerado acima e responda diretamente à pergunta, lembrando de discutir em termos de associação e não causalidade.)*

## Conclusão

*(Escreva a síntese final do trabalho, as recomendações e as limitações encontradas na análise, como por exemplo, problemas de padronização ao unir a base esportiva à do Banco Mundial.)*